In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob, os, shutil, cv2, json, sys
from pathlib import Path

In [ ]:
OPTIONS = json.loads(open('../../Task/info.json', 'r').read())
OPTIONS['dataset'] = Path().resolve().name
OPTIONS

In [ ]:
with open('../../Task/info.json', 'w', encoding='utf-8') as file:
    json.dump(OPTIONS, file, ensure_ascii=False, indent=4)

In [ ]:
SOURCE  = 'files/DataBase.csv'
OUT_DIR = '../../Dataset/'
os.makedirs(OUT_DIR, exist_ok=True)

# CONVERTENDO ANOTAÇÕES

In [ ]:
raw = pd.read_csv(SOURCE)
raw = raw.drop_duplicates(subset='path', keep='last')
print(f'{len(raw)} anotações')
raw.head()

In [ ]:
names  = [os.path.basename(path) for path in raw.path]
labels = [os.path.basename(os.path.dirname(path)) for path in raw.path]

df = pd.DataFrame({
    'path': [os.path.join(OPTIONS['dataset'], 'files', label, name) for label, name in zip(labels, names)],
    'label': labels,
    'x_min': raw.x_min.values, 'y_min': raw.y_min.values,
    'x_max': raw.x_max.values, 'y_max': raw.y_max.values,
})

print(len(df), 'imagens')
df.head()

In [ ]:
missing = [path for path in df.path if not os.path.exists(os.path.join(OUT_DIR, path))]
print('imagens ausentes:', len(missing))
assert len(missing) == 0

df.to_csv(os.path.join(OUT_DIR, 'DataBase.csv'), index=False)
print('salvo em', os.path.join(OUT_DIR, 'DataBase.csv'), '-', len(df), 'linhas')

# CONFERINDO AS CAIXAS

In [ ]:
def showBoxes(ix, ax):
    row = df.iloc[ix]
    img = cv2.cvtColor(cv2.imread(os.path.join(OUT_DIR, row.path)), cv2.COLOR_BGR2RGB)
    cv2.rectangle(img, (int(row.x_min), int(row.y_min)), (int(row.x_max), int(row.y_max)), (255, 0, 0), 8)
    ax.imshow(img)
    ax.set_title(row.label)
    ax.axis('off')


fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, ix in zip(axes, [0, len(df) // 2, len(df) - 1]):
    showBoxes(ix, ax)

plt.tight_layout()
plt.show()